In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pyprojroot import here

In [2]:
ROOT_DIR = here()

RAW_DATA_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"



In [3]:
temp_df = pd.DataFrame()
df = pd.DataFrame()

for filename in os.listdir(RAW_DATA_DIR):
    if filename.endswith(".csv"):
        file_path = os.path.join(RAW_DATA_DIR, filename)
        
        print(f"\n=========================================")
        print(f"FILE: {filename}")
        print(f"=========================================")
        
        # Load the file locally 
        try:
            temp_df = pd.read_csv(file_path, header = None)
            df[filename] = temp_df.iloc[:,0]
            print("data appended to main df")
        except Exception as e:
            print(f"Could not parse file automatically: {e}")


FILE: varna_reefer_timeseries_kW_2024.csv
data appended to main df

FILE: aFRR_timeseries_down_8784_usd_per_kw.csv
data appended to main df

FILE: aFRR_timeseries_up_8784_usd_per_kw.csv
data appended to main df

FILE: Varna_total_annual_hourly_kw_8784.csv
data appended to main df

FILE: Bulgaria_day_ahead_price_2024_usd_per_kwh_clipped.csv
data appended to main df

FILE: varna_port_offices_kw_2024.csv
data appended to main df

FILE: solar_profile_2024_pu.csv
data appended to main df

FILE: varna_container_gantry_power_hourly_2024_total_only.csv
data appended to main df


In [4]:
print(df.shape)
print(df.isna().sum())

df.columns

(8784, 8)
varna_reefer_timeseries_kW_2024.csv                        0
aFRR_timeseries_down_8784_usd_per_kw.csv                   0
aFRR_timeseries_up_8784_usd_per_kw.csv                     0
Varna_total_annual_hourly_kw_8784.csv                      0
Bulgaria_day_ahead_price_2024_usd_per_kwh_clipped.csv      0
varna_port_offices_kw_2024.csv                             0
solar_profile_2024_pu.csv                                  0
varna_container_gantry_power_hourly_2024_total_only.csv    0
dtype: int64


Index(['varna_reefer_timeseries_kW_2024.csv',
       'aFRR_timeseries_down_8784_usd_per_kw.csv',
       'aFRR_timeseries_up_8784_usd_per_kw.csv',
       'Varna_total_annual_hourly_kw_8784.csv',
       'Bulgaria_day_ahead_price_2024_usd_per_kwh_clipped.csv',
       'varna_port_offices_kw_2024.csv', 'solar_profile_2024_pu.csv',
       'varna_container_gantry_power_hourly_2024_total_only.csv'],
      dtype='str')

In [5]:
column_mapping = {
        'varna_reefer_timeseries_kW_2024.csv': 'reefer_demand_kw',
        'aFRR_timeseries_down_8784_usd_per_kw.csv': 'afrr_down_price_usd_kw',
        'aFRR_timeseries_up_8784_usd_per_kw.csv': 'afrr_up_price_usd_kw',
        'Varna_total_annual_hourly_kw_8784.csv': 'total_cold_ironing_demand_kw',
        'Bulgaria_day_ahead_price_2024_usd_per_kwh_clipped.csv': 'day_ahead_price_usd_kwh',
        'varna_port_offices_kw_2024.csv': 'office_demand_kw',
        'solar_profile_2024_pu.csv': 'solar_generation_pu',
        'varna_container_gantry_power_hourly_2024_total_only.csv': 'gantry_crane_demand_kw'
    }

df_renamed = df.rename(columns=column_mapping)

In [6]:
# Define the starting point: January 1, 2024, at 00:00:00
start_datetime = datetime(2024, 1, 1)

# Total hours in the leap year 2024
total_hours = 8784

# Create the dictionary mapping
# Key: Hour index (0 to 8783)
# Value: datetime object for that specific hour
hour_to_datetime_dict = {
    hour: start_datetime + timedelta(hours=hour) 
    for hour in range(total_hours)
}

df_renamed.index = df_renamed.index.map(hour_to_datetime_dict)
df_renamed.head()

,reefer_demand_kw,afrr_down_price_usd_kw,afrr_up_price_usd_kw,total_cold_ironing_demand_kw,day_ahead_price_usd_kwh,office_demand_kw,solar_generation_pu,gantry_crane_demand_kw
2024-01-01 00:00:00,182.6,0.005872,0.00586,0.0,0.000011,24.107,0.0,0.0
2024-01-01 01:00:00,182.6,0.005872,0.00586,350.0,0.000043,24.457,0.0,0.0
2024-01-01 02:00:00,174.4,0.005872,0.00586,350.0,0.000011,24.172,0.0,0.0
2024-01-01 03:00:00,176.1,0.005872,0.00586,350.0,0.000011,24.819,0.0,0.0
2024-01-01 04:00:00,182.6,0.005872,0.00586,350.0,0.000011,25.541,0.0,0.0


In [7]:
df_renamed.to_parquet(f'{PROCESSED_DATA_DIR}/port_energy_data.parquet')

In [9]:
df.shape

(8784, 8)